In [ ]:
%load_ext cudf.pandas

In [ ]:
%%time
### cell 0 ###

import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%time
### cell 1 ###

def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%time
### cell 2 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%time
### cell 3 ###

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%time
### cell 4 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%time
### cell 5 ###

# 1. Filter, compute year & volume, project only needed columns
li = (
    lineitem
    .loc[
        (lineitem.L_SHIPDATE >= "1995-01-01") &
        (lineitem.L_SHIPDATE <  "1997-01-01")
    ]
    .assign(
        L_YEAR=lambda df: df.L_SHIPDATE.dt.year,
        VOLUME=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT)
    )[['L_ORDERKEY','L_SUPPKEY','L_YEAR','VOLUME']]
)

# 2. Pre-filter nation table
nations = nation[
    nation.N_NAME.isin(["FRANCE","GERMANY"])
][['N_NATIONKEY','N_NAME']]

# 3. Attach nation to customer & supplier with minimal columns
cust = (
    customer[['C_CUSTKEY','C_NATIONKEY']]
    .merge(nations, left_on='C_NATIONKEY', right_on='N_NATIONKEY', how='inner')
    [['C_CUSTKEY','N_NAME']]
    .rename(columns={'N_NAME':'CUST_NATION'})
)

supp = (
    supplier[['S_SUPPKEY','S_NATIONKEY']]
    .merge(nations, left_on='S_NATIONKEY', right_on='N_NATIONKEY', how='inner')
    [['S_SUPPKEY','N_NAME']]
    .rename(columns={'N_NAME':'SUPP_NATION'})
)

# 4. Join lineitem → orders → customer → supplier, then filter
df = (
    li
    .merge(orders[['O_ORDERKEY','O_CUSTKEY']], left_on='L_ORDERKEY', right_on='O_ORDERKEY', how='inner')
    .merge(cust, left_on='O_CUSTKEY', right_on='C_CUSTKEY', how='inner')
    .merge(supp, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')
    [['SUPP_NATION','CUST_NATION','L_YEAR','VOLUME']]
)

# 5. Keep only France↔Germany pairs
df = df[
    ((df.SUPP_NATION == 'FRANCE') & (df.CUST_NATION == 'GERMANY')) |
    ((df.SUPP_NATION == 'GERMANY') & (df.CUST_NATION == 'FRANCE'))
]

In [ ]:
%%time
### cell 6 ###

# 6. Aggregate on GPU and rename
total = (
    df
    .groupby(['SUPP_NATION','CUST_NATION','L_YEAR'], as_index=False)['VOLUME']
    .sum()
    .rename(columns={'VOLUME':'REVENUE'})
)